# Diebold-Mariano test: stack_simple_avg vs chronos_price
Load saved predictions from 98a and 04b, merge on (timestamp, symbol), then run DM test for squared-error loss. Tests whether stack_simple_avg is significantly more accurate than chronos_price. Reported **overall** and **per symbol**. Uses HAC variance (Newey-West) and Harvey-Leybourne-Newbold small-sample correction for 7-step forecasts.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

sys.path.insert(0, str(Path.cwd()))
from _pool_common import ARTIFACTS_DIR, FORECAST_HORIZON

path_stack = ARTIFACTS_DIR / "pred_stack_simple_avg_pool.parquet"
path_chronos = ARTIFACTS_DIR / "pred_chronos_pool_price.parquet"
if not path_stack.exists() or not path_chronos.exists():
    raise FileNotFoundError("Run 98a and 04b first to generate pred_*_pool.parquet files.")

pred_stack = pd.read_parquet(path_stack)
pred_chronos = pd.read_parquet(path_chronos)
pred_stack = pred_stack.rename(columns={"y_pred": "y_pred_stack"})
pred_chronos = pred_chronos.rename(columns={"y_pred": "y_pred_chronos"})
merge_cols = ["timestamp", "symbol"]
df = pred_stack[["timestamp", "symbol", "y_true", "y_pred_stack"]].merge(
    pred_chronos[["timestamp", "symbol", "y_pred_chronos"]], on=merge_cols, how="inner"
)
print("Aligned forecasts:", len(df), "rows")
df.head()

Aligned forecasts: 560 rows


,timestamp,symbol,y_true,y_pred_stack,y_pred_chronos
0,2025-12-02,AAPL,286.190002,283.327387,283.135620
1,2025-12-03,AAPL,284.149994,283.405513,283.465393
2,2025-12-04,AAPL,280.700012,283.634083,283.560425
3,2025-12-05,AAPL,278.779999,283.783561,283.301392
4,2025-12-08,AAPL,277.890015,283.886342,283.755371


In [2]:
def dm_test(loss1: np.ndarray, loss2: np.ndarray, h: int = 1):
    """
    Diebold-Mariano test: H0 equal predictive accuracy.
    loss1, loss2: arrays of losses (e.g. squared errors) for the two models.
    h: forecast horizon (for HAC lag and HLN small-sample correction).
    Returns: (dm_stat_adj, p_value_two_tailed, mean_d).
    """
    d = np.asarray(loss1, dtype=float) - np.asarray(loss2, dtype=float)
    n = len(d)
    if n < 2:
        return np.nan, np.nan, np.nan
    mean_d = np.mean(d)
    L = max(1, h - 1)  # Newey-West truncation
    # Autocovariances (biased, 1/n)
    gamma_0 = np.mean((d - mean_d) ** 2)
    var_mean_d = gamma_0
    for j in range(1, min(L + 1, n)):
        gamma_j = np.mean((d[j:] - mean_d) * (d[:-j] - mean_d))
        var_mean_d += 2 * (1 - j / (L + 1)) * gamma_j
    var_mean_d = var_mean_d / n
    var_mean_d = max(var_mean_d, 1e-20)
    dm_stat = mean_d / np.sqrt(var_mean_d)
    # Harvey-Leybourne-Newbold small-sample adjustment
    adj = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    dm_adj = dm_stat * adj
    p_value = 2 * (1 - stats.t.cdf(abs(dm_adj), n - 1))
    return float(dm_adj), float(p_value), float(mean_d)


def run_dm_on_df(sub: pd.DataFrame, h: int, loss: str = "squared"):
    """Compute loss series and run DM; sub has y_true, y_pred_stack, y_pred_chronos."""
    y = sub["y_true"].values
    p1 = sub["y_pred_stack"].values
    p2 = sub["y_pred_chronos"].values
    if loss == "squared":
        loss1 = (y - p1) ** 2
        loss2 = (y - p2) ** 2
    else:
        loss1 = np.abs(y - p1)
        loss2 = np.abs(y - p2)
    return dm_test(loss1, loss2, h=h)


In [3]:
h = int(FORECAST_HORIZON)
dm_adj, p_val, mean_d = run_dm_on_df(df, h)
print("Overall Diebold-Mariano (stack_simple_avg vs chronos_price)")
print("  Loss: squared error. H0: equal predictive accuracy.")
print(f"  DM stat (HLN adj): {dm_adj:.4f}")
print(f"  p-value (two-tailed): {p_val:.4f}")
print(f"  Mean loss diff (stack - chronos): {mean_d:.6f}")
if p_val < 0.05:
    print("  -> Reject H0 at 5%: forecasts differ in accuracy.")
    print("  -> Negative mean_d => stack better; positive => chronos better.")
else:
    print("  -> Do not reject H0 at 5%: no significant difference.")

Overall Diebold-Mariano (stack_simple_avg vs chronos_price)
  Loss: squared error. H0: equal predictive accuracy.
  DM stat (HLN adj): 1.3032
  p-value (two-tailed): 0.1930
  Mean loss diff (stack - chronos): 6.193718
  -> Do not reject H0 at 5%: no significant difference.


In [4]:
rows = []
for sym in sorted(df["symbol"].unique()):
    sub = df[df["symbol"] == sym]
    dm_adj, p_val, mean_d = run_dm_on_df(sub, h)
    rows.append({"symbol": sym, "DM_stat": dm_adj, "p_value": p_val, "mean_loss_diff": mean_d, "n": len(sub)})
dm_per_symbol = pd.DataFrame(rows)
dm_per_symbol["significant_5pct"] = dm_per_symbol["p_value"] < 0.05
dm_per_symbol["better"] = dm_per_symbol["mean_loss_diff"].apply(lambda x: "stack" if x < 0 else "chronos")
print("Per-symbol Diebold-Mariano")
print(dm_per_symbol.to_string(index=False))
dm_per_symbol.to_parquet(ARTIFACTS_DIR / "dm_stack_vs_chronos_price_pool.parquet", index=False)
print("Saved:", ARTIFACTS_DIR / "dm_stack_vs_chronos_price_pool.parquet")

Per-symbol Diebold-Mariano
symbol   DM_stat  p_value  mean_loss_diff  n  significant_5pct  better
  AAPL  1.381893 0.172592       23.792256 56             False chronos
  AMZN  0.149354 0.881821        1.675218 56             False chronos
 GOOGL  0.962309 0.340106       35.480518 56             False chronos
   JNJ -1.783065 0.080094      -10.671427 56             False   stack
   JPM  0.664018 0.509453        4.583537 56             False chronos
  MSFT -0.079525 0.936904       -1.827575 56             False   stack
  NVDA -1.786980 0.079452       -4.434878 56             False   stack
   SPY  1.082427 0.283786       14.843571 56             False chronos
   WMT  1.104254 0.274291        2.406625 56             False chronos
   XOM -1.583139 0.119124       -3.910664 56             False   stack
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\dm_stack_vs_chronos_price_pool.parquet
